# **LaTeX table with the BAO-fit results**

This notebook shows how to create a LaTeX table with the BAO-fit results varying the settings

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["PATH"] = "/global/common/software/nersc9/texlive/2024/bin/x86_64-linux:" + os.environ["PATH"]
os.environ["OMP_NUM_THREADS"] = "1"
import sys
from paths import code_path, save_path
sys.path.append(code_path)
import numpy as np
from itertools import product
from utils_data import GetThetaLimits
from utils_baofit import BAOFitInitializer
from utils_inference import BAOFitChecker
%matplotlib inline

# helper to select only needed keys for a function
def subset(cfg, *keys):
    return {k: cfg[k] for k in keys}

delta_z = 0.02

recon_list = ["pre", "post"]
tracer_list = ["LRG", "LRG+ELG", "ELG"]
tracer_list = ["LRG", "LRG+ELG"]

def get_bins_removed_list(tracer, delta_z):
    if delta_z == 0.02:
        if tracer == "LRG":
            return [
                ("LRG-full", []),
                ("LRG1", list(map(int, np.arange(10, 35)))),
                ("LRG2", list(map(int, np.concatenate([np.arange(0, 10), np.arange(20, 35)])))),
                ("LRG3", list(map(int, np.arange(0, 20))))
            ]
        elif tracer == "LRG+ELG":
            return [(tracer, [])]
        elif tracer == "ELG":
            return [
                ("ELG-full", []),
                ("ELG1", list(map(int, np.arange(15, 40)))),
                ("ELG2", list(map(int, np.arange(0, 15))))
            ]

fit_results = {}
likelihoods = {}
significance_detection = {}

for recon, tracer in product(recon_list, tracer_list):
    dataset = f"DESIY1_{tracer}_{recon}_deltaz{delta_z}"
    bins_removed_list = get_bins_removed_list(tracer, delta_z)

    if tracer == "ELG":
        theta_width = 3
    else:
        theta_width = 6

    theta_min, theta_max = GetThetaLimits(dataset=dataset, nz_flag="mocks", dynamical_theta_limits=True, theta_width=theta_width, code_path=code_path).get_theta_limits()
    
    for tracer_label, bins_removed in bins_removed_list:

        # 1. Arguments
        base_config = {
            "dataset": dataset,
            "weight_type": 1, # it will not be used...
            "mock_id": "mean", # it will not be used...
            "nz_flag": "mocks",
            "cov_type": "mocks",
            "cosmology_template": "desifid",
            "cosmology_covariance": "desifid",
            "delta_theta": 0.2,
            "theta_min": theta_min,
            "theta_max": theta_max,
            "pow_broadband": [-1, 0],
            "bins_removed": bins_removed,
            "alpha_min": 0.8,
            "alpha_max": 1.2,
            # "alpha_type": "alpha_wigg_only",
            "alpha_type": "alpha_wigg_nowigg",
            "code_path": code_path,
            "save_path": save_path,
        }

        baofit_checker = BAOFitChecker(**base_config) # this will print information related to the detection

        if baofit_checker.is_detection:

            config = {
                **base_config,
                "include_wiggles": "",
            }

            # 2. BAO fit initializer. This basically creates the path to load the results
            baofit_initializer = BAOFitInitializer(
                **subset(config, "include_wiggles", "dataset", "weight_type", "mock_id",
                         "nz_flag", "cov_type", "cosmology_template", "cosmology_covariance",
                         "delta_theta", "theta_min", "theta_max", "pow_broadband",
                         "bins_removed", "alpha_min", "alpha_max", "alpha_type", "save_path"),
                verbose=False,
            )

            fit_results[tracer_label, recon] = np.loadtxt(os.path.join(baofit_initializer.path_baofit, "fit_results.txt"))
            likelihoods[tracer_label, recon] = np.loadtxt(os.path.join(baofit_initializer.path_baofit, "likelihood_data.txt"))
            significance_detection[tracer_label, recon] = baofit_checker.significance


In [ ]:
import pandas as pd

rows = []
for (tracer, recon), values in fit_results.items():
    alpha, sigma_alpha, chi2, dof = values

    # format alpha ± sigma
    if sigma_alpha < 100:
        alpha_fmt = f"${alpha:.4f} \\pm {sigma_alpha:.4f}$"
    else:
        alpha_fmt = r"$-$"

    rows.append({
        "tracer": tracer,
        "recon": recon,
        r"$\alpha$": alpha_fmt,
        r"$chi^2$/dof": f"{chi2:.1f}/{int(dof)}",
        r"significance": f"{significance_detection[(tracer, recon)]:.2f}",
    })

df = pd.DataFrame(rows)

latex_table = df.to_latex(index=False)

print(latex_table)
